# 03 - Model Training
Train regression models, tune with RandomizedSearchCV, and save the best model.

In [1]:
import pandas as pd
import numpy as np

import joblib
import json

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

## 1. Load processed data + preprocessor

In [2]:
X_train = pd.read_csv('../data/processed/X_train.csv')
X_test = pd.read_csv('../data/processed/X_test.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').values.ravel()
y_test = pd.read_csv('../data/processed/y_test.csv').values.ravel()

preprocessor = joblib.load('../models/preprocessor.pkl')

X_train_t = preprocessor.transform(X_train)
X_test_t = preprocessor.transform(X_test)

X_train_t.shape, X_test_t.shape

((14844, 118), (3712, 118))

## 2. Tune a model with RandomizedSearchCV

In [3]:
def tune_model(model, param_distributions, X_train, y_train, cv=5, scoring='r2', n_iter=20, random_state=42):
    random_search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_distributions,
        n_iter=n_iter,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        random_state=random_state,
        verbose=1
    )
    random_search.fit(X_train, y_train)
    print(f"Best Params: {random_search.best_params_}")
    print(f"Best CV Score: {random_search.best_score_:.4f}")
    return random_search.best_estimator_, random_search.best_params_, random_search.best_score_

## 3. Param grids

In [4]:
param_distributions = {
    'Ridge': {'alpha': uniform(0.01, 200)},
    'Lasso': {'alpha': uniform(0.001, 20)},
    'DecisionTree': {
        'max_depth': [3, 5, 7, 10, 15, 20, None],
        'min_samples_split': randint(2, 15),
        'min_samples_leaf': randint(1, 8)
    },
    'RandomForest': {
        'n_estimators': randint(100, 400),
        'max_depth': [5, 10, 15, 20, None],
        'min_samples_split': randint(2, 15),
        'min_samples_leaf': randint(1, 6),
        'max_features': ['sqrt', 'log2']
    }
}

## 4. Train Linear Regression (no tuning needed)

In [5]:
lin_reg = LinearRegression()
lin_reg.fit(X_train_t, y_train)

trained_models = {'LinearRegression': lin_reg}
best_params_log = {'LinearRegression': 'N/A'}

## 5. Tune Ridge

In [6]:
ridge_best, ridge_params, ridge_cv = tune_model(
    Ridge(random_state=42), param_distributions['Ridge'], X_train_t, y_train, n_iter=5
)
trained_models['Ridge'] = ridge_best
best_params_log['Ridge'] = ridge_params

Fitting 5 folds for each of 5 candidates, totalling 25 fits
Best Params: {'alpha': np.float64(31.213728088487304)}
Best CV Score: 0.3402


## 6. Tune Lasso

In [7]:
lasso_best, lasso_params, lasso_cv = tune_model(
    Lasso(random_state=42, max_iter=5000), param_distributions['Lasso'], X_train_t, y_train, n_iter=5
)
trained_models['Lasso'] = lasso_best
best_params_log['Lasso'] = lasso_params

Fitting 5 folds for each of 5 candidates, totalling 25 fits
Best Params: {'alpha': np.float64(3.1213728088487303)}
Best CV Score: 0.3428


## 7. Tune Decision Tree

In [8]:
dt_best, dt_params, dt_cv = tune_model(
    DecisionTreeRegressor(random_state=42), param_distributions['DecisionTree'], X_train_t, y_train, n_iter=20
)
trained_models['DecisionTree'] = dt_best
best_params_log['DecisionTree'] = dt_params

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best Params: {'max_depth': 20, 'min_samples_leaf': 5, 'min_samples_split': 13}
Best CV Score: 0.6630


## 8. Tune Random Forest

In [9]:
rf_best, rf_params, rf_cv = tune_model(
    RandomForestRegressor(random_state=42), param_distributions['RandomForest'], X_train_t, y_train, n_iter=25
)
trained_models['RandomForest'] = rf_best
best_params_log['RandomForest'] = rf_params

Fitting 5 folds for each of 25 candidates, totalling 125 fits
Best Params: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 4, 'n_estimators': 314}
Best CV Score: 0.6983


## 9. Save best hyperparameters

In [10]:
with open('../reports/best_params.json', 'w') as f:
    json.dump({k: str(v) for k, v in best_params_log.items()}, f, indent=2)

best_params_log

{'LinearRegression': 'N/A',
 'Ridge': {'alpha': np.float64(31.213728088487304)},
 'Lasso': {'alpha': np.float64(3.1213728088487303)},
 'DecisionTree': {'max_depth': 20,
  'min_samples_leaf': 5,
  'min_samples_split': 13},
 'RandomForest': {'max_depth': None,
  'max_features': 'sqrt',
  'min_samples_leaf': 2,
  'min_samples_split': 4,
  'n_estimators': 314}}

## 10. Evaluate all models on test set

In [11]:
results = []

for name, model in trained_models.items():
    preds = model.predict(X_test_t)
    r2 = r2_score(y_test, preds)
    mae = mean_absolute_error(y_test, preds)
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    results.append({'Model': name, 'R2': r2, 'MAE': mae, 'MSE': mse, 'RMSE': rmse})

results_df = pd.DataFrame(results).sort_values('R2', ascending=False)
results_df

,Model,R2,MAE,MSE,RMSE
4,RandomForest,0.718351,5069.121741,6.164370e+07,7851.350312
3,DecisionTree,0.680499,4901.667701,6.992827e+07,8362.312203
1,Ridge,0.352029,8728.659564,1.418197e+08,11908.808682
0,LinearRegression,0.351489,8750.363194,1.419379e+08,11913.768714
2,Lasso,0.350921,8748.061780,1.420622e+08,11918.985595


In [12]:
results_df.to_csv('../reports/metrics.csv', index=False)

## 11. Pick and save the best model

In [13]:
best_model_name = results_df.iloc[0]['Model']
best_model = trained_models[best_model_name]

print('Best model:', best_model_name)

joblib.dump(best_model, '../models/model.pkl')
print('Saved to ../models/model.pkl')

Best model: RandomForest
Saved to ../models/model.pkl


## 12. Summary

- Trained: Linear Regression, Ridge, Lasso, Decision Tree, Random Forest
- Tuned Ridge/Lasso/Decision Tree/Random Forest with `RandomizedSearchCV` (5-fold CV)
- Best hyperparameters saved to `reports/best_params.json`
- Test-set metrics saved to `reports/metrics.csv`
- Best model (by R²) saved to `models/model.pkl` via joblib

Next notebook: deeper evaluation (`src/evaluate.py`) — actual vs predicted plots, feature importance.